<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Filtering in Frequency Domain — Implementation</b></h1>
</div>

## Setup — Environment and Configuration

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

np.set_printoptions(precision=3, suppress=True)

print("NumPy:", np.__version__)
print("Setup: PASS")

### 0.1 Locate the Lab Automatically


In [ ]:
def locate_lab_root():
    cwd = Path.cwd().resolve()

    for candidate in [cwd, *cwd.parents]:
        if (
            (candidate / "data").is_dir()
            and (candidate / "notebooks" / "main.ipynb").is_file()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the frequency-domain lab root."
    )
LAB_ROOT = locate_lab_root()
DATA_DIR = LAB_ROOT / "data"
OUTPUT_DIR = LAB_ROOT / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
DATASETS = {
    folder.name: sorted(
        path for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    for folder in DATA_DIR.iterdir()
    if folder.is_dir()
}
for group in ("Fourier", "PhaseMag", "Frequency"):
    if not DATASETS.get(group):
        raise FileNotFoundError(
            f"No supported images found in data/{group}/"
        )
def select_input_image(group: str, *keywords: str) -> Path:
    keywords = tuple(keyword.lower() for keyword in keywords)
    matches = [
        path for path in DATASETS[group]
        if all(keyword in path.stem.lower() for keyword in keywords)
    ]
    if not matches:
        raise FileNotFoundError(
            f"No image matching {keywords} found in data/{group}/"
        )
    return matches[0]
print("Lab root:", LAB_ROOT)
print("Data dir:", DATA_DIR)
for group, files in DATASETS.items():
    print(f"{group:10s}: {len(files)} image(s)")
print("Output  :", OUTPUT_DIR)

### 0.2 Reusable helpers

In [ ]:
def load_grayscale_image(path):
    return np.asarray(
        Image.open(path).convert("L"),
        dtype=np.float32
    )
def normalize_to_unit_interval(array):
    array = np.asarray(array, dtype=np.float64)
    lo = array.min()
    hi = array.max()
    if np.isclose(lo, hi):
        return np.zeros_like(array)

    return (array - lo) / (hi - lo)
def display_grayscale_image(ax, image, title, cmap="gray"):
    ax.imshow(image, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
def save_output_figure(fig, filename):
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=160, bbox_inches="tight")
    print("Saved:", path.name)
def compute_centered_fft2(image):
    return np.fft.fftshift(np.fft.fft2(image))
def reconstruct_from_centered_fft2(centered_spectrum):
    return np.real(
        np.fft.ifft2(
            np.fft.ifftshift(centered_spectrum)
        )
    )
def compute_log_spectrum_magnitude(centered_spectrum):
    return np.log1p(np.abs(centered_spectrum))
def compute_fft_roundtrip_tolerance(image, safety_factor=32.0):
    image = np.asarray(image)
    if np.issubdtype(image.dtype, np.floating):
        dtype = image.dtype
    else:
        dtype = np.float64
    eps = np.finfo(dtype).eps

    scale = max(
        1.0,
        float(np.max(np.abs(image)))
    )

    return safety_factor * eps * scale
def mean_squared_error(reference, test):
    reference = np.asarray(
        reference,
        dtype=np.float64
    )
    test = np.asarray(
        test,
        dtype=np.float64
    )

    return np.mean(
        (reference - test) ** 2
    )
def peak_signal_to_noise_ratio(reference, test, peak=255.0):

    error = mean_squared_error(reference, test)
    if np.isclose(error, 0.0):
        return np.inf

    return 10 * np.log10(
        (peak ** 2) / error
    )
def compute_mean_gradient_magnitude(image):
    gy, gx = np.gradient(
        np.asarray(
            image,
            dtype=np.float64
        )
    )

    return np.mean(
        np.hypot(gx, gy)
    )
def measure_ringing_overshoot(image, low=0.0, high=255.0):
    image = np.asarray(
        image,
        dtype=np.float64
    )

    above = max(
        float(image.max() - high),
        0.0
    )
    below = max(
        float(low - image.min()),
        0.0
    )

    return max(above, below)

## 1. Spatial-Frequency Characterization


In [ ]:
width = 512
x = np.linspace(0, 1, width, endpoint=False)

low_signal = np.sin(2 * np.pi * 4 * x)
high_signal = np.sin(2 * np.pi * 32 * x)
low_image = np.tile(low_signal, (180, 1))

high_image = np.tile(high_signal, (180, 1))

fig, axes = plt.subplots(2, 2, figsize=(12, 6))

axes[0, 0].plot(x, low_signal)
axes[0, 0].set_title("4 cycles — low frequency")

display_grayscale_image(
    axes[0, 1],
    low_image,
    "Low spatial frequency"
)

axes[1, 0].plot(x, high_signal)
axes[1, 0].set_title("32 cycles — high frequency")

display_grayscale_image(
    axes[1, 1],
    high_image,
    "High spatial frequency"
)

fig.tight_layout()
save_output_figure(fig, "01_spatial_frequency.png")
plt.show()

## 2. 1-D DFT Validation with Synthetic Sinusoids


In [ ]:
n = np.arange(128)
signal = (
    np.sin(2 * np.pi * 5 * n / len(n))
    + 0.45 * np.sin(2 * np.pi * 18 * n / len(n))
)

signal_spectrum = np.fft.fft(signal)
frequencies = np.fft.fftfreq(len(signal))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(n, signal)
axes[0].set_title("Signal = two sinusoids")

axes[1].stem(
    frequencies[:64],
    np.abs(signal_spectrum[:64])
)
axes[1].set_title("FFT magnitude")
axes[1].set_xlabel("Frequency")

fig.tight_layout()
save_output_figure(fig, "02_fft_1d.png")
plt.show()

## 3. The 2-D Fourier Transform for Images


In [ ]:
HOUSE_PATH = select_input_image("Fourier", "house")
house = load_grayscale_image(HOUSE_PATH)

house_centered_spectrum = compute_centered_fft2(house)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

display_grayscale_image(
    axes[0],
    house,
    f"{HOUSE_PATH.stem} — spatial domain"
)
display_grayscale_image(
    axes[1],
    compute_log_spectrum_magnitude(house_centered_spectrum),
    "Log magnitude spectrum"
)
display_grayscale_image(
    axes[2],
    np.angle(house_centered_spectrum),
    "Phase spectrum",
    cmap="twilight"
)

fig.tight_layout()
save_output_figure(fig, "03_house_spectrum.png")
plt.show()

## 4. 2-D Spectrum Interpretation


In [ ]:
size = 256
coords = np.arange(size)

vertical = np.tile(
    np.sin(2 * np.pi * 16 * coords / size),
    (size, 1)
)

horizontal = vertical.T

xx, yy = np.meshgrid(coords, coords)

diagonal = np.sin(
    2 * np.pi * 12 * (xx + yy) / size
)

patterns = [
    ("Vertical stripes", vertical),
    ("Horizontal stripes", horizontal),
    ("Diagonal stripes", diagonal),
]

fig, axes = plt.subplots(3, 2, figsize=(10, 13))

for row, (name, pattern) in enumerate(patterns):

    F = compute_centered_fft2(pattern)

    display_grayscale_image(
        axes[row, 0],
        pattern,
        name
    )
    display_grayscale_image(
        axes[row, 1],
        compute_log_spectrum_magnitude(F),
        f"{name} — spectrum"
    )

fig.tight_layout()
save_output_figure(fig, "04_orientation_spectra.png")
plt.show()

In [ ]:
frequency_domain_input_files = [
    path for path in DATASETS["Fourier"]
    if path != HOUSE_PATH
]

assert frequency_domain_input_files, "At least one additional Fourier image is required."

fig, axes = plt.subplots(
    len(frequency_domain_input_files),
    2,
    figsize=(12, 4 * len(frequency_domain_input_files)),
    squeeze=False,
)

for row, image_path in enumerate(frequency_domain_input_files):
    image = load_grayscale_image(image_path)
    F = compute_centered_fft2(image)

    display_grayscale_image(
        axes[row, 0],
        image,
        image_path.name
    )
    display_grayscale_image(
        axes[row, 1],
        compute_log_spectrum_magnitude(F),
        f"{image_path.name} — spectrum"
    )

fig.tight_layout()
save_output_figure(fig, "05_dataset_spectra.png")
plt.show()

## 5. Inverse FFT and Reconstruction


In [ ]:
reconstructed_house = reconstruct_from_centered_fft2(
    house_centered_spectrum
)
reconstruction_absolute_error = np.abs(
    house.astype(np.float64)
    - reconstructed_house
)

allowed_reconstruction_error = compute_fft_roundtrip_tolerance(
    house
)

print(
    "Maximum reconstruction error:",
    f"{reconstruction_absolute_error.max():.6e}"
)

print(
    "Float-aware tolerance:",
    f"{allowed_reconstruction_error:.6e}"
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

display_grayscale_image(
    axes[0],
    house,
    "Original"
)
display_grayscale_image(
    axes[1],
    reconstructed_house,
    "IFFT reconstruction"
)
display_grayscale_image(
    axes[2],
    reconstruction_absolute_error,
    "Absolute error"
)

fig.tight_layout()
save_output_figure(fig, "06_reconstruction.png")
plt.show()

## 6. Magnitude–Phase Analysis


In [ ]:
CAT_PATH = select_input_image("PhaseMag", "cat")
WOLF_PATH = select_input_image("PhaseMag", "wolf")

cat = load_grayscale_image(CAT_PATH)
wolf = load_grayscale_image(WOLF_PATH)
common_size = (256, 256)

resized_cat_image = np.asarray(
    Image.fromarray(
        cat.astype(np.uint8)
    ).resize(common_size),
    dtype=np.float32
)

resized_wolf_image = np.asarray(
    Image.fromarray(
        wolf.astype(np.uint8)
    ).resize(common_size),
    dtype=np.float32
)
cat_centered_spectrum = np.fft.fft2(resized_cat_image)
wolf_centered_spectrum = np.fft.fft2(resized_wolf_image)

cat_magnitude_spectrum = np.abs(cat_centered_spectrum)
cat_phase_spectrum = np.angle(cat_centered_spectrum)

wolf_magnitude_spectrum = np.abs(wolf_centered_spectrum)
wolf_phase_spectrum = np.angle(wolf_centered_spectrum)

cat_magnitude_wolf_phase_image = np.real(
    np.fft.ifft2(
        cat_magnitude_spectrum
        * np.exp(1j * wolf_phase_spectrum)
    )
)

wolf_magnitude_cat_phase_image = np.real(
    np.fft.ifft2(
        wolf_magnitude_spectrum
        * np.exp(1j * cat_phase_spectrum)
    )
)

fig, axes = plt.subplots(2, 2, figsize=(10, 10))

display_grayscale_image(axes[0, 0], resized_cat_image, CAT_PATH.stem)
display_grayscale_image(axes[0, 1], resized_wolf_image, WOLF_PATH.stem)
display_grayscale_image(
    axes[1, 0],
    cat_magnitude_wolf_phase_image,
    f"{CAT_PATH.stem} magnitude + {WOLF_PATH.stem} phase"
)
display_grayscale_image(
    axes[1, 1],
    wolf_magnitude_cat_phase_image,
    f"{WOLF_PATH.stem} magnitude + {CAT_PATH.stem} phase"
)

fig.tight_layout()
save_output_figure(fig, "07_phase_magnitude_swap.png")
plt.show()

In [ ]:
magnitude_only = np.real(
    np.fft.ifft2(
        cat_magnitude_spectrum
        * np.exp(
            1j * np.zeros_like(cat_phase_spectrum)
        )
    )
)

phase_only = np.real(
    np.fft.ifft2(
        np.ones_like(cat_magnitude_spectrum)
        * np.exp(1j * cat_phase_spectrum)
    )
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

display_grayscale_image(
    axes[0],
    resized_cat_image,
    "Original"
)
display_grayscale_image(
    axes[1],
    magnitude_only,
    "Magnitude only"
)
display_grayscale_image(
    axes[2],
    phase_only,
    "Phase only"
)

fig.tight_layout()
save_output_figure(
    fig,
    "08_phase_only_magnitude_only.png"
)
plt.show()

## 7. Frequency-Domain Filtering


In [ ]:
def apply_frequency_domain_filter(image, H):
    image = np.asarray(image, dtype=np.float64)
    H = np.asarray(H, dtype=np.float64)
    if image.shape != H.shape:
        raise ValueError(
            f"Image shape {image.shape} and filter shape {H.shape} must match."
        )

    centered_spectrum = compute_centered_fft2(image)

    filtered_spectrum = centered_spectrum * H
    result = reconstruct_from_centered_fft2(filtered_spectrum)

    return result, centered_spectrum, filtered_spectrum
identity_filter = np.ones(
    house.shape,
    dtype=np.float64
)

identity_result, identity_spectrum, identity_filtered_spectrum = (
    apply_frequency_domain_filter(
        house,
        identity_filter
    )
)
identity_error = np.max(
    np.abs(
        identity_result
        - house.astype(np.float64)
    )
)

identity_tolerance = compute_fft_roundtrip_tolerance(
    house
)

assert identity_result.shape == house.shape
assert np.isfinite(identity_result).all()
assert identity_error <= identity_tolerance

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

display_grayscale_image(axes[0], house, "Input image")
display_grayscale_image(
    axes[1],
    compute_log_spectrum_magnitude(identity_spectrum),
    "Centered spectrum"
)
display_grayscale_image(
    axes[2],
    identity_result,
    f"Identity-filter reconstruction\nmax error={identity_error:.2e}"
)

fig.tight_layout()
save_output_figure(fig, "07_frequency_filter_pipeline.png")
plt.show()

print(
    "Frequency-domain pipeline: PASS | "
    f"maximum identity error = {identity_error:.3e} | "
    f"tolerance = {identity_tolerance:.3e}"
)

## 8. Frequency Distance Grid


In [ ]:
def build_frequency_distance_grid(shape):
    rows, cols = shape

    cy = rows // 2
    cx = cols // 2

    y, x = np.ogrid[:rows, :cols]

    return np.sqrt(
        (y - cy) ** 2
        + (x - cx) ** 2
    )
D_house = build_frequency_distance_grid(
    house.shape
)

plt.figure(figsize=(6, 5))
plt.imshow(
    D_house,
    cmap="viridis"
)
plt.title("Distance from frequency origin")
plt.colorbar(
    label="Frequency-bin distance"
)
plt.axis("off")
plt.show()

## 9. Ideal, Gaussian, and Butterworth Low-Pass Filters


In [ ]:
def build_ideal_low_pass_filter(shape, cutoff):
    D = build_frequency_distance_grid(shape)
    return (D <= cutoff).astype(np.float32)
def build_gaussian_low_pass_filter(shape, cutoff):
    D = build_frequency_distance_grid(shape)

    return np.exp(
        -(D ** 2)
        / (2 * cutoff ** 2)
    )
def build_butterworth_low_pass_filter(
    shape,
    cutoff,
    order=2
):
    D = build_frequency_distance_grid(shape)

    return 1.0 / (
        1.0
        + (
            D
            / max(float(cutoff), 1e-12)
        ) ** (2 * order)
    )
cutoff = 30

H_ideal_lp = build_ideal_low_pass_filter(
    house.shape,
    cutoff
)

H_gaussian_lp = build_gaussian_low_pass_filter(
    house.shape,
    cutoff
)

H_butterworth_lp = build_butterworth_low_pass_filter(
    house.shape,
    cutoff,
    order=2
)

house_ideal_lp, _, _ = apply_frequency_domain_filter(
    house,
    H_ideal_lp
)

house_gaussian_lp, _, _ = apply_frequency_domain_filter(
    house,
    H_gaussian_lp
)

house_butterworth_lp, _, _ = apply_frequency_domain_filter(
    house,
    H_butterworth_lp
)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 10)
)

display_grayscale_image(
    axes[0, 0],
    H_ideal_lp,
    "Ideal LPF"
)

display_grayscale_image(
    axes[0, 1],
    H_gaussian_lp,
    "Gaussian LPF"
)

display_grayscale_image(
    axes[0, 2],
    H_butterworth_lp,
    "Butterworth LPF"
)

display_grayscale_image(
    axes[1, 0],
    house_ideal_lp,
    "Ideal result"
)

display_grayscale_image(
    axes[1, 1],
    house_gaussian_lp,
    "Gaussian result"
)

display_grayscale_image(
    axes[1, 2],
    house_butterworth_lp,
    "Butterworth result"
)

fig.tight_layout()
save_output_figure(
    fig,
    "09_lpf_comparison.png"
)
plt.show()
lpf_quality_metrics = {
    "Ideal": {
        "mse": mean_squared_error(
            house,
            house_ideal_lp
        ),
        "psnr": peak_signal_to_noise_ratio(
            house,
            house_ideal_lp
        ),
        "edge_strength": compute_mean_gradient_magnitude(
            house_ideal_lp
        ),
    },
    "Gaussian": {
        "mse": mean_squared_error(
            house,
            house_gaussian_lp
        ),
        "psnr": peak_signal_to_noise_ratio(
            house,
            house_gaussian_lp
        ),
        "edge_strength": compute_mean_gradient_magnitude(
            house_gaussian_lp
        ),
    },
    "Butterworth n=2": {
        "mse": mean_squared_error(
            house,
            house_butterworth_lp
        ),
        "psnr": peak_signal_to_noise_ratio(
            house,
            house_butterworth_lp
        ),
        "edge_strength": compute_mean_gradient_magnitude(
            house_butterworth_lp
        ),
    },
}

In [ ]:
orders = [1, 2, 4, 8]

butterworth_order_results = []

fig, axes = plt.subplots(
    len(orders),
    2,
    figsize=(11, 15)
)

for row, order in enumerate(orders):
    H = build_butterworth_low_pass_filter(
        house.shape,
        cutoff,
        order=order
    )

    result, _, _ = apply_frequency_domain_filter(
        house,
        H
    )

    butterworth_order_results.append({
        "order": order,
        "mse": mean_squared_error(
            house,
            result
        ),
        "edge_strength": compute_mean_gradient_magnitude(
            result
        ),
    })

    display_grayscale_image(
        axes[row, 0],
        H,
        f"Butterworth n={order}"
    )

    display_grayscale_image(
        axes[row, 1],
        result,
        f"Result n={order}"
    )

fig.tight_layout()
save_output_figure(
    fig,
    "10_butterworth_orders.png"
)
plt.show()

## 10. Ringing and the Gibbs Phenomenon


In [ ]:
square = np.zeros(
    (256, 256),
    dtype=np.float32
)

square[64:192, 64:192] = 255.0
ring_cutoff = 22

Hi = build_ideal_low_pass_filter(
    square.shape,
    ring_cutoff
)

Hg = build_gaussian_low_pass_filter(
    square.shape,
    ring_cutoff
)

Hb = build_butterworth_low_pass_filter(
    square.shape,
    ring_cutoff,
    order=2
)

square_i, _, _ = apply_frequency_domain_filter(
    square,
    Hi
)

square_g, _, _ = apply_frequency_domain_filter(
    square,
    Hg
)

square_b, _, _ = apply_frequency_domain_filter(
    square,
    Hb
)

center_row = square.shape[0] // 2

fig, axes = plt.subplots(
    2,
    2,
    figsize=(13, 10)
)

display_grayscale_image(
    axes[0, 0],
    square,
    "Original square"
)

display_grayscale_image(
    axes[0, 1],
    square_i,
    "Ideal LPF — ringing"
)

axes[1, 0].plot(
    square[center_row],
    label="Original"
)

axes[1, 0].plot(
    square_i[center_row],
    label="Ideal"
)

axes[1, 0].plot(
    square_g[center_row],
    label="Gaussian"
)

axes[1, 0].plot(
    square_b[center_row],
    label="Butterworth"
)

axes[1, 0].set_title(
    "Intensity profile through edge"
)
axes[1, 0].legend()

display_grayscale_image(
    axes[1, 1],
    np.abs(square_i - square),
    "Ideal LPF difference"
)

fig.tight_layout()
save_output_figure(
    fig,
    "11_ringing.png"
)
plt.show()
ringing_by_method = {
    "Ideal": measure_ringing_overshoot(
        square_i
    ),
    "Gaussian": measure_ringing_overshoot(
        square_g
    ),
    "Butterworth n=2": measure_ringing_overshoot(
        square_b
    ),
}
butterworth_order_ringing = []

for order in orders:
    H_order = build_butterworth_low_pass_filter(
        square.shape,
        ring_cutoff,
        order=order
    )

    square_order, _, _ = apply_frequency_domain_filter(
        square,
        H_order
    )

    butterworth_order_ringing.append({
        "order": order,
        "ringing": measure_ringing_overshoot(
            square_order
        ),
    })

## 11. High-Pass Filtering


In [ ]:
def build_high_pass_from_low_pass(H_low):
    return 1.0 - H_low
H_ideal_hp = build_high_pass_from_low_pass(
    H_ideal_lp
)

H_gaussian_hp = build_high_pass_from_low_pass(
    H_gaussian_lp
)

H_butterworth_hp = build_high_pass_from_low_pass(
    H_butterworth_lp
)

house_i_hp, _, _ = apply_frequency_domain_filter(
    house,
    H_ideal_hp
)

house_g_hp, _, _ = apply_frequency_domain_filter(
    house,
    H_gaussian_hp
)

house_b_hp, _, _ = apply_frequency_domain_filter(
    house,
    H_butterworth_hp
)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 10)
)

display_grayscale_image(
    axes[0, 0],
    H_ideal_hp,
    "Ideal HPF"
)

display_grayscale_image(
    axes[0, 1],
    H_gaussian_hp,
    "Gaussian HPF"
)

display_grayscale_image(
    axes[0, 2],
    H_butterworth_hp,
    "Butterworth HPF"
)

display_grayscale_image(
    axes[1, 0],
    normalize_to_unit_interval(house_i_hp),
    "Ideal HP response"
)

display_grayscale_image(
    axes[1, 1],
    normalize_to_unit_interval(house_g_hp),
    "Gaussian HP response"
)

display_grayscale_image(
    axes[1, 2],
    normalize_to_unit_interval(house_b_hp),
    "Butterworth HP response"
)

fig.tight_layout()
save_output_figure(
    fig,
    "12_high_pass.png"
)
plt.show()

## 12. High-Boost Sharpening


In [ ]:
high_detail = house_g_hp
high_boost_gains = [0.5, 1.0, 1.5, 2.0]

high_boost_results = []

fig, axes = plt.subplots(
    1,
    len(high_boost_gains) + 1,
    figsize=(4 * (len(high_boost_gains) + 1), 5)
)

display_grayscale_image(
    axes[0],
    house,
    "Original"
)

for col, gain in enumerate(
    high_boost_gains,
    start=1
):
    boosted_unclipped = (
        house.astype(np.float64)
        + gain * high_detail
    )
    clipped_mask = (
        (boosted_unclipped < 0.0)
        | (boosted_unclipped > 255.0)
    )

    boosted = np.clip(
        boosted_unclipped,
        0.0,
        255.0
    )

    clipping_percent = (
        100.0 * np.mean(clipped_mask)
    )

    high_boost_results.append({
        "gain": gain,
        "clipping_percent": clipping_percent,
        "edge_strength": compute_mean_gradient_magnitude(
            boosted
        ),
    })

    display_grayscale_image(
        axes[col],
        boosted,
        (
            f"k={gain:.1f}\n"
            f"clipped={clipping_percent:.2f}%"
        )
    )

fig.tight_layout()
save_output_figure(
    fig,
    "13_high_boost.png"
)
plt.show()
k = 1.2
high_boost = np.clip(
    house.astype(np.float64)
    + k * high_detail,
    0.0,
    255.0
)

## 13. Convolution Theorem


In [ ]:
conv_image = house[:96, :96].astype(np.float64)
conv_kernel = np.ones((5, 5), dtype=np.float64)
conv_kernel /= conv_kernel.sum()

pad_y = conv_kernel.shape[0] // 2
pad_x = conv_kernel.shape[1] // 2

spatial_padded = np.pad(
    conv_image,
    ((pad_y, pad_y), (pad_x, pad_x)),
    mode="constant"
)

windows = np.lib.stride_tricks.sliding_window_view(
    spatial_padded,
    conv_kernel.shape
)

spatial_conv = np.sum(
    windows * conv_kernel[::-1, ::-1],
    axis=(-2, -1)
)
fft_shape = (
    conv_image.shape[0] + conv_kernel.shape[0] - 1,
    conv_image.shape[1] + conv_kernel.shape[1] - 1,
)

fft_full = np.real(
    np.fft.ifft2(
        np.fft.fft2(conv_image, s=fft_shape)
        * np.fft.fft2(conv_kernel, s=fft_shape)
    )
)

fft_conv = fft_full[
    pad_y:pad_y + conv_image.shape[0],
    pad_x:pad_x + conv_image.shape[1]
]
conv_error = np.abs(spatial_conv - fft_conv)

assert spatial_conv.shape == fft_conv.shape
assert np.isfinite(fft_conv).all()
assert conv_error.max() < 1e-8

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

display_grayscale_image(
    axes[0],
    spatial_conv,
    "Direct spatial convolution"
)
display_grayscale_image(
    axes[1],
    fft_conv,
    "Zero-padded FFT convolution"
)
display_grayscale_image(
    axes[2],
    conv_error,
    f"Absolute difference\nmax={conv_error.max():.2e}"
)

fig.tight_layout()
save_output_figure(fig, "13_convolution_validation.png")
plt.show()

print(
    "Convolution validation: PASS | "
    f"maximum absolute difference = {conv_error.max():.3e}"
)

## 14. Band-Pass and Band-Reject Filters


In [ ]:
def build_ideal_band_pass_filter(
    shape,
    low_cutoff,
    high_cutoff
):
    D = build_frequency_distance_grid(shape)

    return (
        (D >= low_cutoff)
        & (D <= high_cutoff)
    ).astype(np.float32)
H_band = build_ideal_band_pass_filter(
    house.shape,
    15,
    55
)

H_band_reject = 1.0 - H_band

house_band, _, _ = apply_frequency_domain_filter(
    house,
    H_band
)

house_band_reject, _, _ = apply_frequency_domain_filter(
    house,
    H_band_reject
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(11, 10)
)

display_grayscale_image(
    axes[0, 0],
    H_band,
    "Band-pass mask"
)

display_grayscale_image(
    axes[0, 1],
    normalize_to_unit_interval(house_band),
    "Band-pass content"
)

display_grayscale_image(
    axes[1, 0],
    H_band_reject,
    "Band-reject mask"
)

display_grayscale_image(
    axes[1, 1],
    house_band_reject,
    "Band-reject result"
)

fig.tight_layout()
save_output_figure(
    fig,
    "14_band_filters.png"
)
plt.show()

## 15. Periodic Interference Analysis


In [ ]:
interference_files = [
    path for path in DATASETS["Frequency"]
    if any(
        token in path.stem.lower()
        for token in ("interference", "moire")
    )
]

assert interference_files, "No periodic-interference images were discovered."

fig, axes = plt.subplots(
    len(interference_files),
    2,
    figsize=(12, 5 * len(interference_files)),
    squeeze=False,
)

for row, image_path in enumerate(interference_files):
    image = load_grayscale_image(image_path)
    F = compute_centered_fft2(image)

    display_grayscale_image(
        axes[row, 0],
        image,
        image_path.name
    )

    display_grayscale_image(
        axes[row, 1],
        compute_log_spectrum_magnitude(F),
        f"{image_path.name} — spectrum"
    )

fig.tight_layout()
save_output_figure(
    fig,
    "15_periodic_noise_spectra.png"
)
plt.show()

## 16. Spectral Peak Detection


In [ ]:
def find_strongest_spectral_peaks(
    centered_spectrum,

number_of_peaks=8,
    center_exclusion_radius=20,
    min_separation=10
):
    magnitude = compute_log_spectrum_magnitude(
        centered_spectrum
    ).copy()

    rows, cols = magnitude.shape
    cy, cx = rows // 2, cols // 2

    yy, xx = np.ogrid[:rows, :cols]
    center_mask = (
        (yy - cy) ** 2
        + (xx - cx) ** 2
        <= center_exclusion_radius ** 2
    )
    magnitude[center_mask] = -np.inf

    flat_order = np.argsort(
        magnitude.ravel()
    )[::-1]

    selected = []

    for flat_index in flat_order:
        y, x = np.unravel_index(
            flat_index,
            magnitude.shape
        )

        separated = all(
            (y - py) ** 2
            + (x - px) ** 2
            >= min_separation ** 2
            for py, px in selected
        )
        if separated:
            selected.append((y, x))
        if len(selected) >= number_of_peaks:
            break

    return selected

## 17. Notch-Reject Filtering


In [ ]:
def build_notch_reject_mask(
    shape,
    peak_locations,
    radius=5
):
    rows, cols = shape
    cy, cx = rows // 2, cols // 2

    yy, xx = np.ogrid[:rows, :cols]
    mask = np.ones(
        shape,
        dtype=np.float32
    )

    for py, px in peak_locations:
        d1 = (
            (yy - py) ** 2
            + (xx - px) ** 2
        )

        mask[d1 <= radius ** 2] = 0.0
        sym_y = 2 * cy - py
        sym_x = 2 * cx - px

        d2 = (
            (yy - sym_y) ** 2
            + (xx - sym_x) ** 2
        )

        mask[d2 <= radius ** 2] = 0.0

    return mask

In [ ]:
ASTRONAUT_PATH = select_input_image("Frequency", "astronaut")
astronaut = load_grayscale_image(ASTRONAUT_PATH)

F_astronaut = compute_centered_fft2(
    astronaut
)
astronaut_peaks = find_strongest_spectral_peaks(
    F_astronaut,
    number_of_peaks=8,
    center_exclusion_radius=25,
    min_separation=12
)
notch_radii = [2, 4, 6, 8]
notch_radius_results = []

astronaut_energy = np.sum(
    np.abs(F_astronaut) ** 2
)

for radius in notch_radii:
    H_radius = build_notch_reject_mask(
        astronaut.shape,
        astronaut_peaks[:4],
        radius=radius
    )

    removed_energy = np.sum(
        np.abs(F_astronaut) ** 2
        * (1.0 - H_radius)
    )

    notch_radius_results.append({
        "radius": radius,
        "removed_energy_percent": (
            100.0
            * removed_energy
            / max(astronaut_energy, 1e-12)
        ),
    })

selected_notch_radius = 4

astronaut_notch = build_notch_reject_mask(
    astronaut.shape,
    astronaut_peaks[:4],
    radius=selected_notch_radius
)

astronaut_filtered = reconstruct_from_centered_fft2(
    F_astronaut
    * astronaut_notch
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(12, 10)
)

display_grayscale_image(axes[0, 0], astronaut, ASTRONAUT_PATH.name)
display_grayscale_image(axes[0, 1], compute_log_spectrum_magnitude(F_astronaut), "Original spectrum")
display_grayscale_image(axes[1, 0], astronaut_notch, f"Notch mask — r={selected_notch_radius}")
display_grayscale_image(axes[1, 1], astronaut_filtered, "After notch filtering")

fig.tight_layout()
save_output_figure(fig, "16_notch_filter.png")
plt.show()

print("Candidate peaks:", astronaut_peaks)

## 18. Moiré Removal


In [ ]:
MOIRE_PATH = select_input_image("Frequency", "moire")
car_moire = load_grayscale_image(MOIRE_PATH)

F_car = compute_centered_fft2(
    car_moire
)

car_peaks = find_strongest_spectral_peaks(
    F_car,
    number_of_peaks=10,
    center_exclusion_radius=30,
    min_separation=12
)

car_notch = build_notch_reject_mask(
    car_moire.shape,
    car_peaks[:6],
    radius=4
)

car_filtered = reconstruct_from_centered_fft2(
    F_car * car_notch
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 5)
)

display_grayscale_image(axes[0], car_moire, MOIRE_PATH.name)
display_grayscale_image(axes[1], compute_log_spectrum_magnitude(F_car), "Moiré spectrum")
display_grayscale_image(axes[2], car_filtered, "Notch-filtered result")

fig.tight_layout()
save_output_figure(
    fig,
    "17_moire_removal.png"
)
plt.show()

## 19. Low-Frequency Illumination Correction


In [ ]:
SPOTSHADE_PATH = select_input_image("Frequency", "spotshade")
spotshade = load_grayscale_image(SPOTSHADE_PATH)

illumination_filter = build_gaussian_low_pass_filter(
    spotshade.shape,
    cutoff=18
)

illumination, _, _ = apply_frequency_domain_filter(
    spotshade,
    illumination_filter
)

epsilon = 1e-6

corrected = spotshade / (
    illumination + epsilon
)

corrected = normalize_to_unit_interval(
    corrected
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

display_grayscale_image(axes[0], spotshade, SPOTSHADE_PATH.name)
display_grayscale_image(axes[1], illumination, "Estimated illumination")
display_grayscale_image(axes[2], corrected, "Normalized image")

fig.tight_layout()
save_output_figure(
    fig,
    "18_shading_correction.png"
)
plt.show()

## 20. Cutoff Sensitivity


In [ ]:
cutoffs = [10, 25, 60, 120]

cutoff_sensitivity_results = []

fig, axes = plt.subplots(
    2,
    len(cutoffs),
    figsize=(4 * len(cutoffs), 8)
)

for col, current_cutoff in enumerate(cutoffs):

    H = build_gaussian_low_pass_filter(
        house.shape,
        current_cutoff
    )

    filtered, _, _ = apply_frequency_domain_filter(
        house,
        H
    )

    cutoff_sensitivity_results.append({
        "cutoff": current_cutoff,
        "mse": mean_squared_error(
            house,
            filtered
        ),
        "edge_strength": compute_mean_gradient_magnitude(
            filtered
        ),
    })

    display_grayscale_image(
        axes[0, col],
        H,
        f"Gaussian LPF\nD0={current_cutoff}"
    )

    display_grayscale_image(
        axes[1, col],
        filtered,
        f"Result\nD0={current_cutoff}"
    )

fig.tight_layout()
save_output_figure(
    fig,
    "19_cutoff_sensitivity.png"
)
plt.show()

## 21. Quantitative Checks


In [ ]:
print(
    "Low-pass comparison at cutoff D0=30"
)

for name, metrics in lpf_quality_metrics.items():
    print(
        f"{name:20s} "
        f"MSE={metrics['mse']:10.3f} "
        f"PSNR={metrics['psnr']:7.2f} dB "
        f"Edge={metrics['edge_strength']:8.3f}"
    )

## 22. Validation Checks


In [ ]:
validation_reconstruction = reconstruct_from_centered_fft2(
    compute_centered_fft2(house)
)
validation_error = np.abs(
    house.astype(np.float64)
    - validation_reconstruction
)

validation_tolerance = compute_fft_roundtrip_tolerance(
    house
)

assert validation_reconstruction.shape == house.shape
assert np.isfinite(validation_reconstruction).all()

assert (
    validation_error.max()
    <= validation_tolerance
), (
    "FFT/IFFT round-trip error exceeded the "
    "dtype-aware tolerance: "
    f"{validation_error.max():.6e} > "
    f"{validation_tolerance:.6e}"
)
filters_to_check = [
    H_ideal_lp,
    H_gaussian_lp,
    H_butterworth_lp,
    H_ideal_hp,
    H_gaussian_hp,
    H_butterworth_hp,
]

for H in filters_to_check:
    assert H.shape == house.shape
    assert np.isfinite(H).all()
    assert H.min() >= 0
    assert H.max() <= 1

center = (
    house.shape[0] // 2,
    house.shape[1] // 2
)

assert np.isclose(
    H_ideal_lp[center],
    1.0
)

assert np.isclose(
    H_gaussian_lp[center],
    1.0
)

assert np.isclose(
    H_butterworth_lp[center],
    1.0
)

assert np.isclose(
    H_ideal_hp[center],
    0.0
)

assert np.isclose(
    H_gaussian_hp[center],
    0.0
)

assert np.isclose(
    H_butterworth_hp[center],
    0.0
)

print(
    "PASS — reconstruction and filter "
    "sanity checks"
)

print(
    "Round-trip max error:",
    f"{validation_error.max():.6e}"
)

print(
    "Accepted tolerance:",
    f"{validation_tolerance:.6e}"
)

## 23. Failure Modes and Diagnostic Signatures


In [ ]:
raw_dynamic_range = (
    np.max(np.abs(house_centered_spectrum))
    / max(
        np.median(np.abs(house_centered_spectrum)),
        1e-12
    )
)

failure_diagnostics = {
    "FFT raw dynamic-range ratio": raw_dynamic_range,
    "FFT/IFFT round-trip max error": validation_error.max(),
    "FFT/IFFT accepted tolerance": validation_tolerance,
    "Convolution max error": conv_error.max(),
    "Ideal LPF ringing overshoot": ringing_by_method["Ideal"],
    "Gaussian LPF ringing overshoot": ringing_by_method["Gaussian"],
    "High-boost max clipping (%)": max(
        row["clipping_percent"]
        for row in high_boost_results
    ),
}

print("Failure-mode diagnostic summary")

for name, value in failure_diagnostics.items():
    print(
        f"  {name:36s}: {value:.6e}"
    )

print()
print(
    "Interpretation: large raw spectral dynamic range "
    "justifies log-magnitude visualization; "
    "round-trip and convolution errors verify numerical "
    "correctness; ringing and clipping quantify the main "
    "artifact risks already observed above."
)

## 24. Parameter Sensitivity and Controlled Experiments


In [ ]:
print("Controlled parameter sensitivity summary")

print("\nGaussian cutoff sweep")
for row in cutoff_sensitivity_results:
    print(
        f"  D0={row['cutoff']:3d} | "
        f"MSE={row['mse']:10.3f} | "
        f"Edge={row['edge_strength']:8.3f}"
    )

print("\nButterworth order sweep")
ringing_lookup = {
    row["order"]: row["ringing"]
    for row in butterworth_order_ringing
}

for row in butterworth_order_results:
    print(
        f"  n={row['order']:2d} | "
        f"MSE={row['mse']:10.3f} | "
        f"Edge={row['edge_strength']:8.3f} | "
        f"Ringing={ringing_lookup[row['order']]:8.3f}"
    )

print("\nHigh-boost gain sweep")
for row in high_boost_results:
    print(
        f"  k={row['gain']:.1f} | "
        f"Clipped={row['clipping_percent']:7.3f}% | "
        f"Edge={row['edge_strength']:8.3f}"
    )

print("\nNotch-radius sweep")
for row in notch_radius_results:
    print(
        f"  r={row['radius']:2d} | "
        f"Removed spectral energy="
        f"{row['removed_energy_percent']:.6f}%"
    )

## 25. Method Selection and Technical Discussion


In [ ]:
method_selection_results = []

for method, metrics in lpf_quality_metrics.items():
    method_selection_results.append({
        "method": method,
        "mse": metrics["mse"],
        "psnr": metrics["psnr"],
        "edge_strength": metrics["edge_strength"],
        "ringing": ringing_by_method[method],
    })

print(
    "Method                 "
    "MSE        PSNR(dB)   "
    "EdgeStrength   Ringing"
)

for row in method_selection_results:
    print(
        f"{row['method']:20s} "
        f"{row['mse']:10.3f} "
        f"{row['psnr']:10.2f} "
        f"{row['edge_strength']:14.3f} "
        f"{row['ringing']:9.3f}"
    )

lowest_change = min(
    method_selection_results,
    key=lambda row: row["mse"]
)

lowest_ringing = min(
    method_selection_results,
    key=lambda row: row["ringing"]
)

strongest_edges = max(
    method_selection_results,
    key=lambda row: row["edge_strength"]
)

print()
print(
    "Lowest numerical change :",
    lowest_change["method"]
)
print(
    "Lowest measured ringing :",
    lowest_ringing["method"]
)
print(
    "Highest retained edge strength:",
    strongest_edges["method"]
)

## 26. Integrated Frequency-Domain Workflow


In [ ]:
def run_frequency_filtering_workflow(
    image_path,
    filter_family="butterworth",
    cutoff=30,
    order=2
):

    image = load_grayscale_image(image_path)
    builders = {
        "ideal": lambda: build_ideal_low_pass_filter(
            image.shape,
            cutoff
        ),
        "gaussian": lambda: build_gaussian_low_pass_filter(
            image.shape,
            cutoff
        ),
        "butterworth": lambda: build_butterworth_low_pass_filter(
            image.shape,
            cutoff,
            order=order
        ),
    }
    if filter_family not in builders:
        raise ValueError(
            "filter_family must be one of: "
            + ", ".join(builders)
        )
    H = builders[filter_family]()

    result, F_input, F_filtered = apply_frequency_domain_filter(
        image,
        H
    )

    assert result.shape == image.shape
    assert H.shape == image.shape
    assert np.isfinite(result).all()
    assert np.isfinite(H).all()
    assert H.min() >= 0.0
    assert H.max() <= 1.0

    report = {
        "image": image_path.name,
        "shape": image.shape,
        "filter_family": filter_family,
        "cutoff": cutoff,
        "order": (
            order

            if filter_family == "butterworth"
            else None
        ),
        "mse_vs_input": mean_squared_error(image, result),
        "psnr_vs_input_db": peak_signal_to_noise_ratio(image, result),
        "filter_min": float(H.min()),
        "filter_max": float(H.max()),
    }

    fig, axes = plt.subplots(1, 5, figsize=(20, 5))

    display_grayscale_image(axes[0], image, "Input")
    display_grayscale_image(
        axes[1],
        compute_log_spectrum_magnitude(F_input),
        "Centered spectrum"
    )
    display_grayscale_image(axes[2], H, "Selected filter")
    display_grayscale_image(
        axes[3],
        compute_log_spectrum_magnitude(F_filtered),
        "Filtered spectrum"
    )
    display_grayscale_image(
        axes[4],
        result,
        "Reconstructed result"
    )

    fig.tight_layout()
    save_output_figure(fig, "23_integrated_workflow.png")
    plt.show()

    return report
final_report = run_frequency_filtering_workflow(
    HOUSE_PATH,
    filter_family="butterworth",
    cutoff=30,
    order=2
)

print("Final workflow configuration")
for key, value in final_report.items():
    print(f"  {key:20s}: {value}")
REQUIRED_OUTPUTS = [
    "01_spatial_frequency.png",
    "02_fft_1d.png",
    "03_house_spectrum.png",
    "04_orientation_spectra.png",
    "05_dataset_spectra.png",
    "06_reconstruction.png",
    "07_phase_magnitude_swap.png",
    "08_phase_only_magnitude_only.png",
    "07_frequency_filter_pipeline.png",
    "09_lpf_comparison.png",
    "10_butterworth_orders.png",
    "11_ringing.png",
    "12_high_pass.png",
    "13_high_boost.png",
    "13_convolution_validation.png",
    "14_band_filters.png",
    "15_periodic_noise_spectra.png",
    "16_notch_filter.png",
    "17_moire_removal.png",
    "18_shading_correction.png",
    "19_cutoff_sensitivity.png",
    "23_integrated_workflow.png",
]
missing_outputs = [
    output_name
    for output_name in REQUIRED_OUTPUTS
    if not (OUTPUT_DIR / output_name).exists()
]
if missing_outputs:
    raise FileNotFoundError(
        "Missing outputs: " + ", ".join(missing_outputs)
    )

print(f"Output validation passed: {len(REQUIRED_OUTPUTS)} files.")
